## Walmart dataset

In [67]:
import pandas as pd
import numpy as np
import plotly.express as px
from sklearn.model_selection import train_test_split, KFold, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.tree import DecisionTreeRegressor
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

1. Store - the store number
2. Date - the week of sales
3. Weekly_Sales - sales for the given store
4. Holiday_Flag - whether the week is a special holiday week 1 – Holiday week 0 – Non-holiday week
5. Temperature - Temperature on the day of sale
6. Fuel_Price - Cost of fuel in the region
7. CPI – Prevailing consumer price index
8. Unemployment - Prevailing unemployment rate 
9. Holiday Events<br /> Super Bowl: 12-Feb-10, 11-Feb-11, 10-Feb-12, 8-Feb-13<br /> Labour Day: 10-Sep-10, 9-Sep-11, 7-Sep-12, 6-Sep-13<br /> Thanksgiving: 26-Nov-10, 25-Nov-11, 23-Nov-12, 29-Nov-13<br /> Christmas: 31-Dec-10, 30-Dec-11, 28-Dec-12, 27-Dec-13

In [68]:
df = pd.read_csv("Walmart.csv")
print(df)

      Store        Date  Weekly_Sales  Holiday_Flag  Temperature  Fuel_Price  \
0         1  05-02-2010    1643690.90             0        42.31       2.572   
1         1  12-02-2010    1641957.44             1        38.51       2.548   
2         1  19-02-2010    1611968.17             0        39.93       2.514   
3         1  26-02-2010    1409727.59             0        46.63       2.561   
4         1  05-03-2010    1554806.68             0        46.50       2.625   
...     ...         ...           ...           ...          ...         ...   
6430     45  28-09-2012     713173.95             0        64.88       3.997   
6431     45  05-10-2012     733455.07             0        64.89       3.985   
6432     45  12-10-2012     734464.36             0        54.47       4.000   
6433     45  19-10-2012     718125.53             0        56.47       3.969   
6434     45  26-10-2012     760281.43             0        58.85       3.882   

             CPI  Unemployment  
0     

## Data Analysis

In [69]:
# Checking for null values
df.isnull().sum()

Store           0
Date            0
Weekly_Sales    0
Holiday_Flag    0
Temperature     0
Fuel_Price      0
CPI             0
Unemployment    0
dtype: int64

In [70]:
# Checking for datatypes
df.dtypes

Store             int64
Date                str
Weekly_Sales    float64
Holiday_Flag      int64
Temperature     float64
Fuel_Price      float64
CPI             float64
Unemployment    float64
dtype: object

In [71]:
# Changing Date (str) to  Datetime
df['Date'] = pd.to_datetime(df['Date'], format='%d-%m-%Y')
df.dtypes


Store                    int64
Date            datetime64[us]
Weekly_Sales           float64
Holiday_Flag             int64
Temperature            float64
Fuel_Price             float64
CPI                    float64
Unemployment           float64
dtype: object

In [72]:
df.describe()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment
count,6435.000000,6435,6.435000e+03,6435.000000,6435.000000,6435.000000,6435.000000,6435.000000
mean,23.000000,2011-06-17 00:00:00,1.046965e+06,0.069930,60.663782,3.358607,171.578394,7.999151
min,1.000000,2010-02-05 00:00:00,2.099862e+05,0.000000,-2.060000,2.472000,126.064000,3.879000
25%,12.000000,2010-10-08 00:00:00,5.533501e+05,0.000000,47.460000,2.933000,131.735000,6.891000
50%,23.000000,2011-06-17 00:00:00,9.607460e+05,0.000000,62.670000,3.445000,182.616521,7.874000
75%,34.000000,2012-02-24 00:00:00,1.420159e+06,0.000000,74.940000,3.735000,212.743293,8.622000
max,45.000000,2012-10-26 00:00:00,3.818686e+06,1.000000,100.140000,4.468000,227.232807,14.313000
std,12.988182,NaN,5.643666e+05,0.255049,18.444933,0.459020,39.356712,1.875885


In [73]:
# Count duplicate rows
duplicate_count = df.duplicated().sum()
print('Number of duplicate rows:', duplicate_count)


Number of duplicate rows: 0


In [74]:
# Splitting the date column
df["Year"] = df["Date"].dt.year
df['Week'] = df['Date'].dt.isocalendar().week
df.head()

,Store,Date,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Week
0,1,2010-02-05,1643690.90,0,42.31,2.572,211.096358,8.106,2010,5
1,1,2010-02-12,1641957.44,1,38.51,2.548,211.242170,8.106,2010,6
2,1,2010-02-19,1611968.17,0,39.93,2.514,211.289143,8.106,2010,7
3,1,2010-02-26,1409727.59,0,46.63,2.561,211.319643,8.106,2010,8
4,1,2010-03-05,1554806.68,0,46.50,2.625,211.350143,8.106,2010,9


In [75]:
# Dropping the date column
df = df.drop(columns=["Date"])
df.head()

,Store,Weekly_Sales,Holiday_Flag,Temperature,Fuel_Price,CPI,Unemployment,Year,Week
0,1,1643690.90,0,42.31,2.572,211.096358,8.106,2010,5
1,1,1641957.44,1,38.51,2.548,211.242170,8.106,2010,6
2,1,1611968.17,0,39.93,2.514,211.289143,8.106,2010,7
3,1,1409727.59,0,46.63,2.561,211.319643,8.106,2010,8
4,1,1554806.68,0,46.50,2.625,211.350143,8.106,2010,9


In [76]:
px.histogram(df, x="Weekly_Sales").show()

In [78]:
# Average weekly sales by stores
# Top 5 stores
print("Top 5 stores")
print(df.groupby('Store')['Weekly_Sales'].mean().sort_values(ascending=False).head())
# Bottom 5 stores
print("Bottom 5 stores")
print(df.groupby('Store')['Weekly_Sales'].mean().sort_values(ascending=True).head())

Top 5 stores
Store
20    2.107677e+06
4     2.094713e+06
14    2.020978e+06
13    2.003620e+06
2     1.925751e+06
Name: Weekly_Sales, dtype: float64
Bottom 5 stores
Store
33    259861.692028
44    302748.866014
5     318011.810490
36    373511.992797
38    385731.653287
Name: Weekly_Sales, dtype: float64


In [79]:
# Weekly sales during holiday (1) and non-holiday weeks (0)
print(print(df.groupby('Holiday_Flag')['Weekly_Sales'].agg(['count', 'mean', 'median', 'min', 'max'])))

              count          mean      median        min         max
Holiday_Flag                                                        
0              5985  1.041256e+06   956211.20  209986.25  3818686.45
1               450  1.122888e+06  1018538.04  215359.21  3004702.33
None


In [80]:
px.imshow(df.corr(), aspect='Auto')

In [82]:
# Splitting the data 
X = df[['Store', 'Holiday_Flag', 'Temperature', 'Fuel_Price',
       'CPI', 'Unemployment', 'Year', 'Week']]
y = df[ 'Weekly_Sales']
# Splitting the data into main data (Train & test) 85% and testing datasets 15%
X_main, X_test, y_main, y_test = train_test_split(X, y, test_size=0.15, random_state=42)
# Splitting the main training data into training &  validation
X_train, X_val, y_train, y_val = train_test_split(X_main, y_main, test_size=0.176, random_state=42) 

In [83]:
# Create regression models
linear_model = LinearRegression()
dt_model = DecisionTreeRegressor(random_state=42)
rf_model = RandomForestRegressor(random_state=42)

In [84]:
# Train the models
linear_model.fit(X_train, y_train)
dt_model.fit(X_train, y_train)
rf_model.fit(X_train, y_train)


,"random_state random_state: int, RandomState instance or None, default=NoneControls both the randomness of the bootstrapping of the samples usedwhen building trees (if ``bootstrap=True``) and the sampling of thefeatures to consider when looking for the best split at each node(if ``max_features < n_features``).See :term:`Glossary <random_state>` for details.",42
,"n_estimators n_estimators: int, default=100The number of trees in the forest... versionchanged:: 0.22 The default value of ``n_estimators`` changed from 10 to 100 in 0.22.",100
,"criterion criterion: {""squared_error"", ""absolute_error"", ""poisson""}, default=""squared_error""The function to measure the quality of a split. Supported criteriaare ""squared_error"" for the mean squared error, which is equal tovariance reduction as feature selection criterion and minimizes the L2loss using the mean of each terminal node, ""absolute_error"" for the meanabsolute error, which minimizes the L1 loss using the median of each terminalnode, and ""poisson"" which uses reduction in Poisson deviance to find splits,also using the mean of each terminal node... versionadded:: 0.18 Mean Absolute Error (MAE) criterion... versionadded:: 1.0 Poisson criterion... versionchanged:: 1.9 Criterion `""friedman_mse""` was deprecated.",'squared_error'
,"max_depth max_depth: int, default=NoneThe maximum depth of the tree. If None, then nodes are expanded untilall leaves are pure or until all leaves contain less thanmin_samples_split samples.",None
,"min_samples_split min_samples_split: int or float, default=2The minimum number of samples required to split an internal node:- If int, then consider `min_samples_split` as the minimum number.- If float, then `min_samples_split` is a fraction and `ceil(min_samples_split * n_samples)` are the minimum number of samples for each split... versionchanged:: 0.18 Added float values for fractions.",2
,"min_samples_leaf min_samples_leaf: int or float, default=1The minimum number of samples required to be at a leaf node.A split point at any depth will only be considered if it leaves atleast ``min_samples_leaf`` training samples in each of the left andright branches. This may have the effect of smoothing the model,especially in regression.- If int, then consider `min_samples_leaf` as the minimum number.- If float, then `min_samples_leaf` is a fraction and `ceil(min_samples_leaf * n_samples)` are the minimum number of samples for each node... versionchanged:: 0.18 Added float values for fractions.",1
,"min_weight_fraction_leaf min_weight_fraction_leaf: float, default=0.0The minimum weighted fraction of the sum total of weights (of allthe input samples) required to be at a leaf node. Samples haveequal weight when sample_weight is not provided.",0.0
,"max_features max_features: {""sqrt"", ""log2"", None}, int or float, default=1.0The number of features to consider when looking for the best split:- If int, then consider `max_features` features at each split.- If float, then `max_features` is a fraction and `max(1, int(max_features * n_features_in_))` features are considered at each split.- If ""sqrt"", then `max_features=sqrt(n_features)`.- If ""log2"", then `max_features=log2(n_features)`.- If None or 1.0, then `max_features=n_features`... note:: The default of 1.0 is equivalent to bagged trees and more randomness can be achieved by setting smaller values, e.g. 0.3... versionchanged:: 1.1 The default of `max_features` changed from `""auto""` to 1.0.Note: the search for a split does not stop until at least onevalid partition of the node samples is found, even if it requires toeffectively inspect more than ``max_features`` features.",1.0
,"max_leaf_nodes max_leaf_nodes: int, default=NoneGrow trees with ``max_leaf_nodes`` in best-first fashion.Best nodes are defined as relative reduction in impurity.If None then unlimited number of leaf nodes.",None
,"min_impurity_decrease min_impurity_decrease: float, default=0.0A node will be split if this split induces a decrease o

In [85]:
# Make predictions using each model
linear_pred = linear_model.predict(X_val)
dt_pred = dt_model.predict(X_val)
rf_pred = rf_model.predict(X_val)

# Create a function to calculate model performance
def evaluate_model(model_name, y_test, y_pred):
    mae = mean_absolute_error(y_test, y_pred)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    r2 = r2_score(y_test, y_pred)
    
    return {
        'Model': model_name,
        'MAE': mae,
        'RMSE': rmse,
        'R-squared': r2
    }
# Store results in a list
results = [
    evaluate_model('Linear Regression', y_val, linear_pred),
    evaluate_model('Decision Tree Regressor', y_val, dt_pred),
    evaluate_model('Random Forest Regressor', y_val, rf_pred)
]

# Convert results into a DataFrame
results_df = pd.DataFrame(results)

results_df

,Model,MAE,RMSE,R-squared
0,Linear Regression,429099.988600,515367.034925,0.138072
1,Decision Tree Regressor,67679.493427,115848.881046,0.956447
2,Random Forest Regressor,55612.883374,95578.807694,0.970354


In [86]:
test_results = [
    evaluate_model(
        'Linear Regression',
        y_test,
        linear_model.predict(X_test)
    ),
    evaluate_model(
        'Decision Tree Regressor',
        y_test,
        dt_model.predict(X_test)
    ),
    evaluate_model(
        'Random Forest Regressor',
        y_test,
        rf_model.predict(X_test)
    )
]

test_results_df = pd.DataFrame(test_results)
print(test_results_df)

                     Model            MAE           RMSE  R-squared
0        Linear Regression  433138.104857  520513.017730   0.156237
1  Decision Tree Regressor   80232.695093  154321.625544   0.925833
2  Random Forest Regressor   63673.335544  119260.157006   0.955706


### Markdown for Regression results
The random forest regressor performed better in both test and validation prediction, so we could choose that as best model